In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

In [4]:
# Подключаем Google Диск к Colab
from google.colab import drive
drive.mount('/content/drive')

attendance_enriched = pd.read_csv('/content/drive/MyDrive/проект аналитика/attendance_enriched_new.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
# 1. Анализ всех групп в ИТИАД
print("="*60)
print("АНАЛИЗ ГРУПП ИТИАД")
print("="*60)

# Фильтруем только ИТИАД
itiad_df = attendance_enriched[attendance_enriched['Faculty'] == 'Институт информационных технологий и анализа данных'].copy()

# Извлекаем базовые названия групп
itiad_df['Group_Base'] = itiad_df['Group'].str.extract(r'([A-Za-zА-Яа-я]+)')[0]

# Анализируем все группы в ИТИАД
group_stats = itiad_df.groupby('Group_Base').agg({
    'mira_id': 'nunique',
    'id': 'count'
}).rename(columns={'mira_id': 'unique_students', 'id': 'total_records'}).sort_values('unique_students', ascending=False)

print("\nВсе группы ИТИАД с количеством студентов:")
print(group_stats)
print(f"\nВсего различных групп: {len(group_stats)}")

# 2. Выбираем группу для анализа
TARGET_GROUP = 'АСУб'

print(f"\n" + "="*60)
print(f"АНАЛИЗ ГРУППЫ: {TARGET_GROUP}")
print("="*60)

# 3. Фильтруем данные по выбранной группе
group_df = itiad_df[itiad_df['Group_Base'] == TARGET_GROUP].copy()

# Статистика по группе
print(f"\nСтатистика по группе {TARGET_GROUP}:")
print(f"- Всего записей: {len(group_df)}")
print(f"- Уникальных студентов: {group_df['mira_id'].nunique()}")
print(f"- Варианты названий групп: {group_df['Group'].unique()}")
print(f"- Года групп: {sorted(group_df['Group'].str.extract(r'(\d+)')[0].dropna().unique())}")

# Проверяем, что студентов достаточно для обучения
if group_df['mira_id'].nunique() < 10:
    print(f"\nВНИМАНИЕ: Группа {TARGET_GROUP} содержит мало студентов ({group_df['mira_id'].nunique()})")
    print("Рекомендуется выбрать другую группу с большим количеством студентов")


АНАЛИЗ ГРУПП ИТИАД

Все группы ИТИАД с количеством студентов:
            unique_students  total_records
Group_Base                                
ИСТб                    154          45652
АСУб                    132          31963
ИБб                     104          22844
ЭВМб                     87          18167
ИСИб                     64          10736
КСм                      24           1414
ИИТм                     22           1455
ИТПм                     17            619
БКСм                     15           1287
ЦППм                      7            326

Всего различных групп: 10

АНАЛИЗ ГРУППЫ: АСУб

Статистика по группе АСУб:
- Всего записей: 31963
- Уникальных студентов: 132
- Варианты названий групп: ['АСУб-21' 'АСУб-22' 'АСУб-24' 'АСУб-23']
- Года групп: ['21', '22', '23', '24']


In [8]:
# ============================================================
# ОБУЧЕНИЕ МОДЕЛИ ДЛЯ ВЫБРАННОЙ ГРУППЫ
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Функция для нормализации оценок
def normalize_grade(grade):
    if pd.isna(grade):
        return np.nan
    if isinstance(grade, str):
        grade_lower = grade
        if 'Зачтено' in grade_lower and 'Не зачтено' not in grade_lower:
            return 5.0
        elif 'Не зачтено' in grade_lower or 'Н/Я' in grade_lower:
            return 2.0
        else:
            try:
                clean_grade = ''.join(c for c in grade if c.isdigit() or c == '.' or c == ',')
                clean_grade = clean_grade.replace(',', '.')
                return float(clean_grade)
            except:
                return np.nan
    else:
        try:
            return float(grade)
        except:
            return np.nan

# Добавляем числовой столбец с оценкой
group_df['grade_numeric'] = group_df['Result'].apply(normalize_grade)

# Удаляем строки с некорректными оценками
print(f"\nИсходное количество записей: {len(group_df)}")
group_df = group_df.dropna(subset=['grade_numeric'])
print(f"После удаления некорректных оценок: {len(group_df)}")

# Определяем курс студента по группе
def group_to_course(group_name):
    """Определяет курс по названию группы"""
    # Извлекаем год из названия группы
    year_match = re.search(r'(\d+)', str(group_name))
    if year_match:
        year = int(year_match.group(1))
        # Предполагаем, что 24 год - 1 курс, 23 - 2 курс, 22 - 3 курс, 21 - 4 курс
        if year >= 24:
            return 1
        elif year == 23:
            return 2
        elif year == 22:
            return 3
        elif year == 21:
            return 4
    return None

group_df['student_course'] = group_df['Group'].apply(group_to_course)

# Создаем признаки для каждого студента
print("\nСоздаем расширенные признаки для студентов...")

student_features = []

for mira_id in group_df['mira_id'].unique():
    student_data = group_df[group_df['mira_id'] == mira_id]

    # Определяем курс (берем первый попавшийся, предполагаем, что у студента он один)
    student_course = student_data['student_course'].iloc[0] if not student_data['student_course'].isna().all() else None

    # Базовые признаки
    grades = student_data['grade_numeric'].values

    avg_grade = np.mean(grades)
    std_grade = np.std(grades) if len(grades) > 1 else 0
    min_grade = np.min(grades)
    max_grade = np.max(grades)
    num_subjects = len(grades)

    # Процент оценок в разных диапазонах
    excellent = np.sum(grades >= 4.5) / num_subjects if num_subjects > 0 else 0
    good = np.sum((grades >= 4.0) & (grades < 4.5)) / num_subjects if num_subjects > 0 else 0
    satisfactory = np.sum((grades >= 3.0) & (grades < 4.0)) / num_subjects if num_subjects > 0 else 0
    poor = np.sum(grades < 3.0) / num_subjects if num_subjects > 0 else 0

    # Количество различных оценок
    unique_grades = len(np.unique(grades))

    student_features.append({
        'mira_id': mira_id,
        'student_course': student_course,
        'avg_grade': avg_grade,
        'std_grade': std_grade,
        'min_grade': min_grade,
        'max_grade': max_grade,
        'num_subjects': num_subjects,
        'excellent_rate': excellent,
        'good_rate': good,
        'satisfactory_rate': satisfactory,
        'poor_rate': poor,
        'unique_grades': unique_grades,
        'group': student_data['Group'].iloc[0]
    })

# Создаем DataFrame с признаками
features_df = pd.DataFrame(student_features)

print(f"Создано {len(features_df)} записей с признаками")




Исходное количество записей: 31963
После удаления некорректных оценок: 31963

Создаем расширенные признаки для студентов...
Создано 132 записей с признаками


In [9]:
# ============================================================
# ДОБАВЛЕНИЕ ПРИЗНАКОВ ПОСЕЩАЕМОСТИ
# ============================================================

print("\nДобавляем признаки посещаемости...")

# Берём уникальные пары студент-предмет
unique_attendance = group_df[['mira_id', 'всего_занятий', 'посещений', 'процент_посещений']].drop_duplicates()

# Агрегируем по студентам
attendance_features = unique_attendance.groupby('mira_id').agg(
    total_lessons_all=('всего_занятий', 'sum'),
    total_attended_all=('посещений', 'sum'),
    avg_subject_attendance=('процент_посещений', 'mean'),
    std_subject_attendance=('процент_посещений', 'std'),
    min_subject_attendance=('процент_посещений', 'min'),
    max_subject_attendance=('процент_посещений', 'max'),
    num_subjects_with_attendance=('всего_занятий', 'count')
).reset_index()

# Общий процент посещаемости
attendance_features['overall_attendance_percent'] = (
    attendance_features['total_attended_all'] / attendance_features['total_lessons_all'] * 100
).round(1)

attendance_features.fillna(0, inplace=True)

# Объединяем с features_df
features_df = features_df.merge(attendance_features, on='mira_id', how='left')

# Заполняем нулями студентов без данных посещаемости
attendance_cols = [
    'total_lessons_all', 'total_attended_all', 'avg_subject_attendance',
    'std_subject_attendance', 'min_subject_attendance', 'max_subject_attendance',
    'num_subjects_with_attendance', 'overall_attendance_percent'
]
features_df[attendance_cols] = features_df[attendance_cols].fillna(0)

print("Признаки посещаемости добавлены.")
print(features_df.head())




Добавляем признаки посещаемости...
Признаки посещаемости добавлены.
     mira_id  student_course  avg_grade  std_grade  min_grade  max_grade  \
0  2418744.0               4   3.464286   1.295105        2.0        5.0   
1  2425907.0               4   3.158730   1.382502        2.0        5.0   
2  2421197.0               3   4.317073   0.829507        3.0        5.0   
3  2445089.0               3   4.609971   0.854737        2.0        5.0   
4  2441444.0               3   4.831776   0.374065        4.0        5.0   

   num_subjects  excellent_rate  good_rate  satisfactory_rate  ...  \
0            28        0.392857   0.000000           0.285714  ...   
1            63        0.317460   0.095238           0.015873  ...   
2           123        0.552846   0.211382           0.235772  ...   
3           341        0.774194   0.143695           0.000000  ...   
4           321        0.831776   0.168224           0.000000  ...   

   unique_grades    group total_lessons_all  total_at

In [10]:
# ============================================================
# ПОДГОТОВКА ДАННЫХ ДЛЯ МОДЕЛИ
# ============================================================

print("\nПодготовка данных для модели предсказания...")

# Разделяем студентов по курсам
course1 = features_df[features_df['student_course'] == 1]
course2 = features_df[features_df['student_course'] == 2]
course3 = features_df[features_df['student_course'] == 3]
course4 = features_df[features_df['student_course'] == 4]

print(f"Студентов 1 курса: {len(course1)}")
print(f"Студентов 2 курса: {len(course2)}")
print(f"Студентов 3 курса: {len(course3)}")
print(f"Студентов 4 курса: {len(course4)}")

# Признаки для модели
feature_cols = [
    'avg_grade', 'std_grade', 'min_grade', 'max_grade',
    'num_subjects', 'excellent_rate', 'good_rate',
    'satisfactory_rate', 'poor_rate', 'unique_grades',
    'total_lessons_all', 'total_attended_all', 'overall_attendance_percent',
    'avg_subject_attendance', 'std_subject_attendance',
    'min_subject_attendance', 'max_subject_attendance',
    'num_subjects_with_attendance'
]

# Подготовка данных (студенты 2-4 курсов для обучения)
X = []
y = []

for _, student in features_df.iterrows():
    if student['student_course'] is not None and student['student_course'] > 1:
        features = [student[col] for col in feature_cols]
        X.append(features)
        y.append(student['avg_grade'])

X = np.array(X)
y = np.array(y)

print(f"Размерность X: {X.shape}")
print(f"Размерность y: {y.shape}")

# Проверка, достаточно ли данных для обучения
if len(X) < 10:
    print("\nВНИМАНИЕ: Недостаточно данных для обучения модели!")
    print(f"Доступно только {len(X)} примеров. Нужно минимум 10-20.")
    print("Рекомендуется выбрать группу с большим количеством студентов.")

# Нормализация
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Разделение на train/test
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print(f"Тренировочные примеры: {X_train.shape[0]}")
print(f"Тестовые примеры: {X_test.shape[0]}")

# Dataset класс
class StudentDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y).unsqueeze(1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Модель
class GradeRegressor(nn.Module):
    def __init__(self, input_size):
        super(GradeRegressor, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_size, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1)
        )

    def forward(self, x):
        return self.model(x)

# Инициализация
input_size = X_train.shape[1]
model = GradeRegressor(input_size)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# DataLoader
train_dataset = StudentDataset(X_train, y_train)
test_dataset = StudentDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# Обучение
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

num_epochs = 200
train_losses = []
val_losses = []

print("\nНачинаем обучение модели...")

for epoch in range(num_epochs):
    # Train
    model.train()
    train_loss = 0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        predictions = model(batch_X)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            predictions = model(batch_X)
            loss = criterion(predictions, batch_y)
            val_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(test_loader)
    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)

    if (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")

# Оценка модели
def evaluate_model(model, dataloader, device='cpu'):
    model.eval()
    all_predictions = []
    all_targets = []

    with torch.no_grad():
        for batch_X, batch_y in dataloader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            predictions = model(batch_X)
            all_predictions.extend(predictions.cpu().numpy().flatten())
            all_targets.extend(batch_y.cpu().numpy().flatten())

    mae = mean_absolute_error(all_targets, all_predictions)
    mse = mean_squared_error(all_targets, all_predictions)
    rmse = np.sqrt(mse)
    r2 = r2_score(all_targets, all_predictions)

    return {
        'MAE': mae,
        'MSE': mse,
        'RMSE': rmse,
        'R2': r2,
        'predictions': all_predictions,
        'targets': all_targets
    }

metrics = evaluate_model(model, test_loader, device)

print("\n" + "="*60)
print(f"МЕТРИКИ КАЧЕСТВА МОДЕЛИ ДЛЯ ГРУППЫ {TARGET_GROUP}:")
print("="*60)
print(f"MAE: {metrics['MAE']:.3f}")
print(f"RMSE: {metrics['RMSE']:.3f}")
print(f"R²: {metrics['R2']:.3f}")

# Сохранение результатов
output_filename = f'{TARGET_GROUP}_student_features.csv'
features_df.to_csv(output_filename, index=False, encoding='utf-8')
print(f"\nПризнаки студентов сохранены в '{output_filename}'")


Подготовка данных для модели предсказания...
Студентов 1 курса: 46
Студентов 2 курса: 43
Студентов 3 курса: 39
Студентов 4 курса: 4
Размерность X: (86, 18)
Размерность y: (86,)
Тренировочные примеры: 68
Тестовые примеры: 18

Начинаем обучение модели...
Epoch [20/200], Train Loss: 7.1332, Val Loss: 7.3576
Epoch [40/200], Train Loss: 1.8029, Val Loss: 1.1936
Epoch [60/200], Train Loss: 1.6469, Val Loss: 0.7572
Epoch [80/200], Train Loss: 1.3919, Val Loss: 0.3851
Epoch [100/200], Train Loss: 1.1483, Val Loss: 0.3654
Epoch [120/200], Train Loss: 1.0244, Val Loss: 0.4410
Epoch [140/200], Train Loss: 0.9705, Val Loss: 0.2972
Epoch [160/200], Train Loss: 0.7787, Val Loss: 0.2942
Epoch [180/200], Train Loss: 1.0906, Val Loss: 0.2232
Epoch [200/200], Train Loss: 0.4939, Val Loss: 0.2321

МЕТРИКИ КАЧЕСТВА МОДЕЛИ ДЛЯ ГРУППЫ АСУб:
MAE: 0.342
RMSE: 0.394
R²: 0.453

Признаки студентов сохранены в 'АСУб_student_features.csv'
